# 09b · Complementary evidence for Q3 (Stage 3D, exploratory)

## Purpose
Stage 3C.1 left two problems in the unknown-fault decision: withheld faults passed off as a **known fault** (mostly
imbalance ↔ vertical misalignment) and withheld faults declared **healthy** (mostly horizontal misalignment). This notebook
evaluates three candidate additions, computed only from saved scores (no neural-network training), on the same 27 LOCO
runs × seeds 42/43/44.

> **Exploratory development results on one partition. The candidates were chosen after seeing 3C.1 results.** Any final
> claim uses the existing reserved-recording protocol (the `loco_*` and other reserved-test columns in `metadata.csv`),
> opened once at the end. All nine fault classes have informed development decisions. Reserved recordings test those
> decisions; they do not establish performance on entirely new fault families.

## Candidates (fixed before running; CLAUDE.md §7)
- **E1 deep ensemble** (one predictor per run): the three seeds' recording probabilities averaged (ens_msp, ensemble
  label); ens_maha_rank = mean over seeds of the Mahalanobis distance's empirical rank against that seed's calibration
  distances; seed disagreement (Jensen–Shannon) reported only. Both checks pass → healthy or that fault; else unknown.
- **E2 LR + CNN combined unfamiliarity** (per seed): s = max(r(1 − LR MSP), r(CNN Mahalanobis)); the CNN MSP check is
  unchanged; s replaces rule B's Mahalanobis check.
- **E3 B1s healthy veto** (per seed): B1s refitted per run on the run's AE fit normals (vibration-only features);
  **D** = rule B + healthy needs B1s ≤ τ_b; **D′** = D + healthy also needs AE ≤ τ_e.

Ranks: r(x) = #{calibration ≤ x} / (n + 1). Thresholds: the 3C order-statistic rule on calibration (97.5 %); τ_e and τ_b
= maximum calibration-normal score (2–3 normals per run). References: rules A, B, C of 3C.1. No AE override in E1, E2.

## Pre-declared success criteria (pass / fail; no selection)
- **Problem 1:** fewer withheld instances passed as known than rule B, with the known-fault false-unknown macro rates on
  (a) and (b) each at most 2 points above rule B's.
- **Problem 2:** fewer withheld instances called healthy than rule B, shown with the change in normal instances sent to
  unknown. No ratio or cost conclusion.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the Stage 3D evaluation (tables, criteria, threshold-free comparisons) and stops if it was produced under different split assignments.</div>

In [ ]:
import json

import pandas as pd

from ml.cache import check_split_hash
from ml.paths import SYSTEM_CODE_DIR

meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
RES = SYSTEM_CODE_DIR / "results" / "loco" / "complementary"
ev = json.loads((RES / "evaluation.json").read_text())
check_split_hash(ev["provenance"]["split_hash"], meta, "evaluation.json")
tab = pd.read_csv(RES / "evaluation_table.csv")
print(ev["provenance"]["wording"])

## 1. Candidates against rules A / B / C
Instance counts (withheld instances = distinct recordings; fold-k normals are 225 instances of 25 distinct healthy
recordings; seen-folder normals 74 instances). Per seed and the seed mean; E1 appears once. False-unknown rates on known
faults are macro over known fault classes.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows, for every method and seed (and the seed mean), the withheld outcomes, the known-fault false-unknown macro rates and wrong-fault counts for (a) and (b), and the normal instances declared healthy.</div>

In [ ]:
cols = ["method", "seed", "withheld_unknown", "withheld_known_fault", "withheld_healthy", "a_unknown_macro",
        "a_wrong_inst", "b_unknown_macro", "b_wrong_inst", "normal_foldk_healthy", "normal_foldk_unknown",
        "normal_seen_healthy", "normal_seen_unknown"]
order = {"A": 0, "B": 1, "C": 2, "E1 (ensemble, once)": 3, "E2": 4, "D": 5, "D′": 6}
t = tab[cols].assign(o=tab.method.map(order), s=tab.seed.astype(str)).sort_values(["o", "s"]).drop(columns=["o", "s"])
t.set_index(["method", "seed"]).round(3)

### Withheld outcomes per class (seed 42 and E1)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For rule B (seed 42), E1 and each seed-42 candidate: per withheld class, the instances called unknown, passed as a known fault and declared healthy.</div>

In [ ]:
def per_class(key):
    return pd.DataFrame(ev["summaries"][key]["withheld_per_class"]).T[["unknown", "known_fault", "healthy"]]


pd.concat({k: per_class(k) for k in ("B_s42", "E1", "E2_s42", "D_s42", "D′_s42")}, axis=1)

## 2. Pre-declared criteria (pass / fail)

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates Problem 1 and Problem 2 for every candidate against rule B of the same seed, against the seed mean, and E1 against the seed mean and each seed; with the numbers behind each verdict.</div>

In [ ]:
rows = []
for k, v in ev["criteria"].items():
    p1, p2 = v["problem1_detail"], v["problem2_detail"]
    rows.append({"candidate": k, "Problem 1": "PASS" if v["problem1_pass"] else "fail",
                 "withheld → known (cand., B)": p1["withheld_known_fault"],
                 "(a) FU − B, points": round(p1["a_unknown_macro_minus_B_pp"], 2),
                 "(b) FU − B, points": round(p1["b_unknown_macro_minus_B_pp"], 2),
                 "Problem 2": "PASS" if v["problem2_pass"] else "fail",
                 "withheld → healthy (cand., B)": p2["withheld_healthy"],
                 "normal → unknown change, fold k": p2["normal_to_unknown_change_foldk"],
                 "normal → unknown change, seen": p2["normal_to_unknown_change_seen"]})
print(ev["criteria_definition"])
pd.DataFrame(rows).set_index("candidate")

## 3. Threshold-free comparisons (paired, 3C.1 bootstrap)
Per run AUROC / OSCR, withheld vs fold-k known, for −ens_msp and ens_maha_rank (E1, against each seed's CNN Mahalanobis)
and s (E2, seed-matched). One stratified mixed-unit draw over the development pool is applied to all 27 runs (recording
multiplicities); runs without withheld or known instances in a draw are dropped from that draw. **An interval excluding
zero supports a specific metric difference under this protocol, not an overall winner.**

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates each paired comparison: macro values of both scores, the macro difference, its bootstrap interval, and in how many of the 27 runs the candidate score is higher.</div>

In [ ]:
tf = ev["threshold_free"]
print(tf["wording"]); print(tf["bootstrap"])
pd.DataFrame(tf["comparisons"]).T.assign(ci95=lambda d: d.ci95.map(lambda c: f"[{c[0]:+.3f}, {c[1]:+.3f}]")).round(3)

## 4. Supporting measurements

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows E2's measured joint acceptance (calibration, seen folder, unseen fold k), B1s's false alarms on normal instances with the calibration-normal counts, and the number of rule-B caution flags (healthy verdict with AE error above τ_e; B_with_caution makes the same decisions as B).</div>

In [ ]:
acc = pd.read_csv(RES / "e2_joint_acceptance_per_run.csv")
display(acc.groupby(["seed", "group"])[["msp_acceptance", "s_acceptance", "joint_acceptance"]].mean().round(3))
fa = pd.read_csv(RES / "e3_b1s_false_alarms_per_run.csv")
display(fa.groupby("group")[["false_alarms", "n"]].sum().assign(calibration_normals_per_run=str(sorted(fa.n_cal_normals.unique()))))
pd.DataFrame({k: v for k, v in ev["summaries"].items() if k.startswith("B_with_caution")}).T

## 5. Stage ML-4 corrections (3D results unchanged)
**E2 interpretation.** In 67 of 81 runs, τ_s equals the maximum value the rank formula can produce, and ties are accepted, so the combined-score check rejects no recording in those runs; only the MSP check rejects. This follows the agreed rank-and-threshold design. E2 therefore does not test whether combining LR and CNN evidence helps; any redesigned combination would be a separately declared experiment.

**Seed disagreement** (the declared E1 score, reported only; not in any rule): threshold-free AUROC per withheld class,
withheld vs fold-k known, mean over folds — *computed after the Stage 3D report; descriptive*.

**D.** D reduces unfamiliar faults called healthy (seed 42: 19 → 2) at a cost in healthy acceptance (184 → 157 of 225 instances from 25 distinct recordings; τ_b from 2–3 calibration normals per run). Vibration-only B1s alone had 6/25 false alarms in the Q1 evaluation. Candidate only; not selected.

**Q3.** The tested scores and rules do not consistently distinguish the imbalance / vertical-misalignment pair when one class is withheld. The classifier distinguishes them well when both are represented in training. This does not establish that no representation, training objective or decision method could help. The confusion is not attributed to missing cross-axis information; that is a hypothesis only.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows, per seed, how many runs have a non-restrictive τ_s (equal to the largest possible rank) and how many recordings the combined-score check rejected there, then the AUROC of the seed disagreement per withheld class.</div>

In [ ]:
add = json.loads((RES / "addenda_ml4.json").read_text())
nr = pd.read_csv(RES / "addenda_e2_tau_s_per_run.csv")
display(nr.groupby("seed").agg(runs=("non_restrictive", "size"), non_restrictive=("non_restrictive", "sum"),
                               s_check_rejections_in_those_runs=("rejected_by_s_check_eval",
                                                                 lambda x: int(x[nr.loc[x.index, "non_restrictive"]].sum()))))
print(add["disagreement_auroc"]["label"])
pd.Series(add["disagreement_auroc"]["per_class_mean_over_folds"], name="AUROC (mean over folds)").round(3).to_frame().assign(
    macro=round(add["disagreement_auroc"]["macro"], 3))

## Findings
Exploratory development results on one partition; the candidates were chosen after seeing the 3C.1 results. Pass/fail
only; nothing is selected.
- **Q3 (ML-4 wording).** The tested scores and rules do not consistently distinguish the imbalance / vertical-misalignment pair when one class is withheld. The classifier distinguishes them well when both are represented in training. This does not establish that no representation, training objective or decision method could help.
- **Problem 1 (withheld → known fault) is not solved by any candidate.** Rule B passes 283 / 269 / 261 withheld instances
  as known (seeds 42 / 43 / 44). E1 passes 275 (fail against the seed mean of 271; passes only against seed 42). E2 passes
  more (336 / 319 / 336) although it lowers false unknowns on known faults. D and D′ leave the known-fault decisions
  unchanged by design (283 / 269 / 261). The imbalance ↔ vertical-misalignment confusion remains in every candidate
  (e.g. E1: imbalance → known 142 of 144, vertical misalignment → known 105 of 125).
- **Problem 2 (withheld → healthy): D and D′ pass for every seed.** Withheld instances called healthy: B 19 / 17 / 24,
  D 2 / 1 / 0, D′ 1 / 0 / 0. In exchange, more normal instances become unknown: D +27 / +27 / +24 fold-k and +8 / +11 / +8
  seen-folder instances (repeats of 25 distinct healthy recordings); D′ +38 / +39 / +33 and +14 / +17 / +13. E1 passes
  against the seed mean by one instance (19 vs 20); E2 fails (38 / 42 / 34). No ratio or cost conclusion is drawn.
- **D (ML-4 wording).** D reduces unfamiliar faults called healthy (seed 42: 19 → 2) at a cost in healthy acceptance (184 → 157 of 225 instances from 25 distinct recordings; τ_b from 2–3 calibration normals per run). Vibration-only B1s alone had 6/25 false alarms in the Q1 evaluation. Candidate only; not selected.
- **E2 (ML-4 wording).** In 67 of 81 runs, τ_s equals the maximum value the rank formula can produce, and ties are accepted, so the combined-score check rejects no recording in those runs; only the MSP check rejects. This follows the agreed rank-and-threshold design. E2 therefore does not test whether combining LR and CNN evidence helps; any redesigned combination would be a separately declared experiment.
- **Seed disagreement (computed after the Stage 3D report; descriptive):** AUROC withheld vs fold-k known, mean over
  folds, macro 0.86; imbalance 0.52 and vertical misalignment 0.58; the other seven classes 0.87–0.98.
- **The B1s veto rests on a weak threshold.** τ_b uses 2–3 calibration normals per run; B1s alone raises false alarms on
  53 of 225 fold-k and 16 of 74 seen-folder normal instances.
- **Threshold-free (paired, 3C.1 bootstrap):** E1's ensemble scores do not differ from a single seed's CNN Mahalanobis
  (all intervals include zero). E2's s is higher by +0.010 / +0.011 / +0.024 macro AUROC but in only 7–8 of 27 runs;
  only seed 44's OSCR interval excludes zero ([+0.001, +0.046]), which supports that specific difference under this
  protocol, not an overall winner.
- **Seed disagreement (E1, reported only):** median Jensen–Shannon divergence 0.044 on withheld recordings vs 0.0001 on
  every other group.


## Outputs written
(by `scripts/09c_loco_complementary.py`) `results/loco/complementary/*.json|csv`;
`results/scores/e1_ensemble_s42-43-44_loco_decisions.parquet`, `e2_lr_cnn_rank_s<seed>_loco_decisions.parquet`,
`e3_b1s_veto_s<seed>_loco_decisions.parquet`, `b1s_nomic_loco_q3fit_scores.parquet`.